In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import xarray as xr
import healpy as hp
import gc
import sys

sys.path.append('../')
import lonboard
from utils.healpix_plot import *

In [2]:
# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [3]:
ds_healpix = xr.open_dataset("/home/ubuntu/project/sentinel-2-dggs-ai-processor/src/notebook/healpix_10m.zarr", engine='zarr')
available_cell_ids = ds_healpix.cell_ids.values
print(f"Number of available HEALPix cells: {len(available_cell_ids):,}")

Number of available HEALPix cells: 2,158,232


In [4]:
level = 19
band_list = ds_healpix.bands.values
in_channels = len(band_list)
out_channels = len(band_list)
stride = 1
print(f"Model config: {in_channels} input channels → {out_channels} output channels")

Model config: 4 input channels → 4 output channels


In [5]:
class NeighborIndexProcessor:
    """External processor for building neighbor indices - OUTSIDE the model"""

    def __init__(self, level, nest=True):
        self.level = level
        self.NSIDE = 2 ** level
        self.nest = nest
        self.cache = {}

    def build_neighbor_indices(self, available_cell_ids, stride=1):
        """Build neighbor indices for given cell IDs and stride"""
        # Create cache key
        cache_key = (tuple(sorted(available_cell_ids)), stride)

        if cache_key in self.cache:
            return self.cache[cache_key]

        available_cell_set = set(available_cell_ids)
        neighbor_indices = []

        # Create cell_id to data_index mapping
        cell_to_data_idx = {cell_id: i for i, cell_id in enumerate(available_cell_ids)}

        # Apply stride to center cell list
        center_cells = available_cell_ids[::stride]

        for cell_id in center_cells:
            neighbors = hp.get_all_neighbours(self.NSIDE, cell_id, nest=self.nest)


            # Validate each neighbor; replace invalid or missing with center
            valid_neighbors = [
                n if (n != -1 and n in available_cell_set) else cell_id
                for n in neighbors
            ]

            patch = [cell_id] + valid_neighbors  # Center + 8 neighbors

            # Convert to data indices
            patch_data_indices = [cell_to_data_idx[cell_id] for cell_id in patch]
            neighbor_indices.append(patch_data_indices)

        # Convert to tensor
        data_neighbor_indices = torch.tensor(neighbor_indices, dtype=torch.long)

        # Cache the result
        self.cache[cache_key] = data_neighbor_indices

        return data_neighbor_indices

    def clear_cache(self):
        """Clear the cache"""
        self.cache.clear()

In [6]:
print("Creating neighbor processor...")
neighbor_processor = NeighborIndexProcessor(level=level, nest=True)
neighbor_indices = neighbor_processor.build_neighbor_indices(available_cell_ids, stride)
print(f"Generated {neighbor_indices.shape[0]:,} patches")

Creating neighbor processor...
Generated 2,158,232 patches


In [7]:
import xdggs

ds_healpix = ds_healpix.pipe(xdggs.decode)

In [8]:
# Use of tanh to concentrate the scale variation for the lower values
lonboard.Map(
    [
        # exploire_layer(
        #     ds_healpix.Sentinel2.sel(bands=band_list[0]).compute(),
        #     alpha=0.10,
        #     cmap='viridis'
        # ),
        exploire_layer(
            ds_healpix.Sentinel2.sel(bands=band_list[-1]).compute()[neighbor_indices[100]],
            alpha=1,
            cmap='plasma'
        ),
    ]
)

Map(custom_attribution='', layers=(SolidPolygonLayer(filled=True, get_fill_color=arro3.core.ChunkedArray<Fixed…

In [9]:
patchIndex = 100
p1 = neighbor_indices[patchIndex]

neighbor_order = [4, 1, 2, 3, 0, 5, 6, 7, 8]  # SW, W, NW, N, center, NE , E, SE, S
patch = ds_healpix.Sentinel2.sel(bands=band_list[-1]).compute()[p1].cell_ids.values[neighbor_order].reshape(3, 3)
patch

array([[185450552167, 185450552161, 185450552163],
       [185450552166, 185450552164, 185450552165],
       [185450552143, 185450552142, 185450552139]])

## 2D Convolution for 1 patch index of chunk product 

In [10]:
# Reshape the patch for 2D convolution with right elements order

neighbor_order = [4, 1, 2, 3, 0, 5, 6, 7, 8]  # SW, W, NW, N, center, NE , E, SE, S
patch = ds_healpix.Sentinel2.sel(bands=band_list[-1]).compute()[p1].values[neighbor_order].reshape(3, 3)
# transform the patch to a tensor
patch = torch.tensor(patch, dtype=torch.float32)
# Unsqueeze to add batch and channel dimensions
patch = patch.unsqueeze(0).unsqueeze(0)
# Define a 2D convolution layer
# with a 3x3 kernel, stride of 1, and no bias
conv2d = nn.Conv2d(1, 1, stride=1, kernel_size=3, bias=True)

In [11]:
# Apply conv2d to kernel (treating kernel as input data)
conv_output = conv2d(patch)
conv_result = conv_output.item()

[W807 17:41:01.779879140 NNPACK.cpp:57] Could not initialize NNPACK! Reason: Unsupported hardware.


In [12]:
print(f"Conv2d input: {patch.shape}")
print(f"Conv2d output: {conv_output.shape}")
print(f"Conv2d result: {conv_result:.6f}")

Conv2d input: torch.Size([1, 1, 3, 3])
Conv2d output: torch.Size([1, 1, 1, 1])
Conv2d result: 0.103475


In [13]:
# Manual dot product (element-wise multiplication + sum) + bias from conv2d
# This is equivalent to the conv2d operation for a single patch
dot_product = torch.sum(patch * conv2d.weight) + conv2d.bias.item()
print(f"Conv2d result: {dot_product:.6f}")

Conv2d result: 0.103475


In [14]:
class SphericalConv(nn.Module):

    def __init__(self, in_channels, out_channels, bias=True):
        super(SphericalConv, self).__init__()

        # 2D convolution for 3x3 patches
        self.conv = nn.Conv2d(in_channels, out_channels, stride=1, kernel_size=3, padding=0, bias=bias)

        # # Initialize weights
        # nn.init.kaiming_normal_(self.conv.weight)
        # if bias:
        #     nn.init.constant_(self.conv.bias, 0.0)

    def forward(self, x, neighbor_indices, neighbor_order=None):

        batch_size, n_channels, n_cells = x.shape
        n_patches = neighbor_indices.shape[0]

        # Move neighbor_indices to same device as input
        if neighbor_indices.device != x.device:
            neighbor_indices = neighbor_indices.to(x.device)

        neighbor_order = [8, 1, 2,   # NW, N, NE  (top row)
                          7, 0, 3,   # W, center, E  (middle row)
                          6, 5, 4]   # SW, S, SE  (bottom row)


        patches = x[:, :, neighbor_indices]
        patches_reordered = patches[:, :, :, neighbor_order]
        patches_2d = patches_reordered.view(batch_size * n_patches, n_channels, 3, 3)
        output = self.conv(patches_2d)
        output = output.squeeze(-1).squeeze(-1)  # Remove H, W dims
        output = output.view(batch_size, -1, n_patches)  #
        return output

In [15]:
model = SphericalConv(in_channels, out_channels, bias=False)

# Load and prepare data
spectral_data = []
for band in band_list:
    band_data = ds_healpix.Sentinel2.sel(bands=band).compute().values
    spectral_data.append(band_data)

In [16]:
x_multi_band = np.stack(spectral_data, axis=0)
x_tensor = torch.tensor(x_multi_band, dtype=torch.float32).unsqueeze(0)
print(f"Input tensor shape: {x_tensor.shape}")
print("Running forward pass...")
model.eval()
with torch.no_grad():
    output = model(x_tensor, neighbor_indices)
    print(f"Output tensor shape: {output.shape}")

Input tensor shape: torch.Size([1, 4, 2158232])
Running forward pass...
Output tensor shape: torch.Size([1, 4, 2158232])


In [ ]:
class SphericalConvBlock(nn.Module):
    """
    Complete Spherical Convolution block with BatchNorm, Activation, and Pooling.

    This block processes HEALPix data through:
    1. Spherical convolution (extracts features from 3x3 neighborhoods)
    2. Batch normalization (stabilizes training)
    3. Activation function (introduces non-linearity)
    4. Optional dropout (prevents overfitting)
    5. 1D pooling (reduces spatial resolution along patches dimension)

    The key insight is that after SphericalConv, we have:
    [batch, out_channels, n_patches] - a 1D sequence of patch features
    So we need 1D operations (BatchNorm1d, MaxPool1d) not 2D operations.
    """

    def __init__(self, in_channels, out_channels, activation='relu', use_dropout=False, dropout_rate=0.1, pool_size=2):
        super(SphericalConvBlock, self).__init__()

        self.conv = SphericalConv(in_channels, out_channels)

        self.bn = nn.BatchNorm1d(out_channels)
        if activation.lower() == 'relu':
            self.activation = nn.ReLU()
        elif activation.lower() == 'gelu':
            self.activation = nn.GELU()
        elif activation.lower() == 'swish':
            self.activation = nn.SiLU()
        elif activation.lower() == 'leaky_relu':
            self.activation = nn.LeakyReLU(0.2)
        else:
            self.activation = nn.ReLU()
        self.use_dropout = use_dropout
        if use_dropout:
            self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x, neighbor_indices, neighbor_order=None):
        """
        Forward pass through the spherical convolution block.

        Parameters
        ----------
        x : torch.Tensor
            Input tensor of shape [batch_size, in_channels, n_cells]
        neighbor_indices : torch.Tensor
            Neighbor indices of shape [n_patches, 9]
        neighbor_order : list, optional
            Order for arranging 9 neighbors into 3x3 grid
        Returns
        -------
        torch.Tensor
            Output tensor of shape [batch_size, out_channels, n_patches//pool_size]
        """
        # Spherical convolution: [B, C_in, N_cells] → [B, C_out, N_patches]
        x = self.conv(x, neighbor_indices, neighbor_order)
        # bach Normalization: [B, C_out, N_patches] → [B, C_out, N_patches]
        x = self.bn(x)
        # Activation: [B, C_out, N_patches] → [B, C_out, N_patches]
        x = self.activation(x)

        # Optional dropout: [B, C_out, N_patches] → [B, C_out, N_patches]
        if self.use_dropout:
            x = self.dropout(x)

        return x

In [25]:
model = SphericalConvBlock(in_channels, out_channels)
print(f"Input tensor shape: {x_tensor.shape}")
model.eval()
with torch.no_grad():
    output = model(x_tensor, neighbor_indices)
    print(f"Output tensor shape: {output.shape}")

Input tensor shape: torch.Size([1, 4, 2158232])
Output tensor shape: torch.Size([1, 4, 2158232])


In [47]:
class SphericalDoubleConvBlock(nn.Module):
    """
    """

    def __init__(self, in_channels, out_channels):
        super(SphericalDoubleConvBlock, self).__init__()

        self.conv1 = SphericalConvBlock(in_channels, out_channels)
        self.conv2 = SphericalConvBlock(out_channels, out_channels)
        self.pool = nn.MaxPool1d(kernel_size=4)

    def forward(self, x, neighbor_indices):
        """
        """
        x = self.conv1(x, neighbor_indices)
        x = self.conv2(x, neighbor_indices)
        x = self.pool(x)
        return x

In [48]:
model = SphericalDoubleConvBlock(in_channels=4, out_channels=4)
print(f"Input tensor shape: {x_tensor.shape}")
model.eval()
with torch.no_grad():
    output = model(x_tensor, neighbor_indices)
    print(f"Output tensor shape: {output.shape}")

Input tensor shape: torch.Size([1, 4, 2158232])
Output tensor shape: torch.Size([1, 4, 539558])


In [ ]:
2158232/4

539558.0

In [36]:
output = output.squeeze(0)
output = output.cpu().numpy()
print(f"Output tensor shape after squeeze: {output.shape}")
output

Output tensor shape after squeeze: (4, 2158232)


array([[0.        , 0.        , 0.08702101, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.08691613, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.08715423, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.0876001 , ..., 0.        , 0.        ,
        0.        ]], shape=(4, 2158232), dtype=float32)

In [38]:
var_cell_ids = xr.DataArray(
    available_cell_ids,
    dims="cells",
    attrs={
        "grid_name": "healpix",
        "indexing_scheme": "nested" ,
        "resolution": 19,
    }
)

ds_output = xr.DataArray(
            output,
            dims=("bands", "cells"),
             coords={
                "bands": band_list,
                "cell_ids": var_cell_ids
            },
            name='Sentinel2',
           )

ds_total = ds_output.to_dataset()

In [39]:
ds_total = ds_total.pipe(xdggs.decode)

In [40]:
sys.path.append('../')
from data.healpix import get_chunk_info, get_bands, get_chunk, healpix_projection
from utils.plot import plot_all_chunks
from utils.healpix_plot import exploire_layer

import lonboard
from utils.healpix_plot import *

lonboard.Map(
    [
        exploire_layer(
            ds_total.Sentinel2.sel(bands=band_list[-1]).compute(),
            alpha=0.80,
            cmap='viridis'
        )

    ]
)

Map(custom_attribution='', layers=(SolidPolygonLayer(filled=True, get_fill_color=arro3.core.ChunkedArray<Fixed…

In [52]:
# =============================================================================
# HEALPix Parent-Child Utility Functions
# =============================================================================

import healpy as hp

def get_parent_cell_ids(child_cell_ids, child_level, parent_level, nest=True):
    """
    Get parent HEALPix cell IDs from child cell IDs at different resolution levels.

    Parameters
    ----------
    child_cell_ids : array-like
        Array of HEALPix cell IDs at the child (higher) resolution level.
    child_level : int
        Resolution level of the child cells (higher number = finer resolution).
    parent_level : int
        Target parent resolution level (lower number = coarser resolution).
    nest : bool, optional
        Whether to use nested HEALPix indexing scheme. Default: True.

    Returns
    -------
    numpy.ndarray
        Array of parent cell IDs corresponding to each input child cell.

    Examples
    --------
    >>> # Get level 18 parents from level 19 cells
    >>> child_ids = np.array([185450551767, 185450551772, 185450551776])
    >>> parent_ids = get_parent_cell_ids(child_ids, child_level=19, parent_level=18)
    >>> print(parent_ids)

    Notes
    -----
    - parent_level must be less than child_level (coarser resolution)
    - Uses bit-shifting for nested indexing: parent_id = child_id >> (2 * level_diff)
    - For ring indexing, converts between nested and ring schemes as needed
    - Maintains same indexing scheme (nested/ring) for output as specified
    """
    child_cell_ids = np.asarray(child_cell_ids)

    # Validate inputs
    if parent_level >= child_level:
        raise ValueError(f"Parent level ({parent_level}) must be less than child level ({child_level})")

    if parent_level < 0 or child_level < 0:
        raise ValueError("Resolution levels must be non-negative")

    level_diff = child_level - parent_level
    child_nside = 2 ** child_level
    parent_nside = 2 ** parent_level

    if nest:
        # For nested indexing, use bit-shifting (fast method)
        parent_cell_ids = child_cell_ids >> (2 * level_diff)
    else:
        # For ring indexing, convert through nested scheme
        # Ring -> Nested -> Parent Nested -> Parent Ring
        child_nested = hp.ring2nest(child_nside, child_cell_ids)
        parent_nested = child_nested >> (2 * level_diff)
        parent_cell_ids = hp.nest2ring(parent_nside, parent_nested)

    return parent_cell_ids

def get_all_parent_levels(child_cell_ids, child_level, nest=True):
    """
    Get parent cell IDs for all resolution levels from 0 to child_level-1.

    Parameters
    ----------
    child_cell_ids : array-like
        Array of HEALPix cell IDs at the finest resolution level.
    child_level : int
        Resolution level of the input cells.
    nest : bool, optional
        Whether to use nested HEALPix indexing scheme. Default: True.

    Returns
    -------
    dict
        Dictionary mapping parent levels to arrays of parent cell IDs.
        Keys are levels 0 to child_level-1, values are numpy arrays.

    Examples
    --------
    >>> child_ids = np.array([185450551767, 185450551772])
    >>> all_parents = get_all_parent_levels(child_ids, child_level=19)
    >>> print(f"Level 18 parents: {all_parents[18]}")
    >>> print(f"Level 0 parent: {all_parents[0]}")  # Should be 0-11 (base HEALPix cells)
    """
    result = {}

    for parent_level in range(child_level):
        result[parent_level] = get_parent_cell_ids(
            child_cell_ids, child_level, parent_level, nest=nest
        )

    return result


In [50]:
print("Creating neighbor processor Level 19...")
neighbor_processor = NeighborIndexProcessor(level=level, nest=True)
neighbor_indices_level_19 = neighbor_processor.build_neighbor_indices(available_cell_ids, stride)
print(f"Generated {neighbor_indices_level_19.shape[0]:,} patches")
neighbor_indices_level_19[:10]

Creating neighbor processor Level 19...
Generated 2,158,232 patches


tensor([[  0,   0,   1,   2, 265, 259, 257,   0,   0],
        [  1,   1,   1,   3,   4,   2,   0,   1,   1],
        [  2,   1,   3,   4, 267, 265, 259,   0,   2],
        [  3,   3,  10,  13,  14,   4,   2,   1,   3],
        [  4,   3,  13,  14, 289, 267, 265,   2,   1],
        [  5,   5,   5,   7,   8,   6,   5,   5,   5],
        [  6,   5,   7,   8,  19,  17,  11,   6,   6],
        [  7,   7,  65,  68,  69,   8,   6,   5,   7],
        [  8,   7,  68,  69,  80,  19,  17,   6,   5],
        [  9,   9,   9,  11,  12,  10,   9,   9,   9]])

In [58]:
2158232/4

539558.0

In [53]:
level_18_parent_ids = get_parent_cell_ids(available_cell_ids, child_level=19, parent_level=18, nest=True)
available_cell_ids_18 = np.unique(level_18_parent_ids)

In [56]:
len(available_cell_ids_18)

540592

In [ ]:
print("Creating neighbor processor Level 18...")
neighbor_processor = NeighborIndexProcessor(level=18, nest=True)
neighbor_indices_level18 = neighbor_processor.build_neighbor_indices(available_cell_ids_18, stride)
print(f"Generated {neighbor_indices_level18.shape[0]:,} patches")
neighbor_indices_level18[:10]

In [ ]:
# pool of square window of size=3, stride=2
m = nn.MaxPool2d(2)

input = torch.randn(1, 4, 30, 30)
output = m(input)

In [ ]:
output.shape

In [ ]:
# Basic 2D CNN: Conv2D + ReLU + Pool
neighbor_order = [4, 1, 2, 3, 0, 5, 6, 7, 8]  # SW, W, NW, N, center, NE , E, SE, S
patch = ds_healpix.Sentinel2.sel(bands=band_list[-1]).compute()[p1].values[neighbor_order].reshape(3, 3)
# transform the patch to a tensor
patch = torch.tensor(patch, dtype=torch.float32)
# Unsqueeze to add batch and channel dimensions
patch = patch.unsqueeze(0).unsqueeze(0)

# Define a 2D convolution layer
conv2d = nn.Conv2d(1, 1, stride=1, kernel_size=3, bias=True)
# ReLU activation
relu = nn.ReLU()
# 2D pooling (we'll use a 1x1 pool since our input becomes 1x1 after conv)
pool = nn.AdaptiveAvgPool2d(1)

print(f"Input patch shape: {patch.shape}")
print(f"Input patch:\n{patch.squeeze()}")

# Apply conv2d
conv_output = conv2d(patch)
print(f"After Conv2D: {conv_output.shape}")

# Apply ReLU
relu_output = relu(conv_output)
print(f"After ReLU: {relu_output.shape}")
print(f"After ReLU: {relu_output}")


# Apply pooling
pool_output = pool(relu_output)
print(f"After Pooling: {pool_output.shape}")
print(f"Final output: {pool_output.item():.6f}")